In [1]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.7 MB/s eta 0:00:00


In [2]:
import os
import copy
import time
import math
import random
import zipfile
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from PIL import Image
from ultralytics import YOLO

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

print("torch:", torch.__version__)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
torch: 2.11.0+cu128


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA A100-SXM4-40GB


In [5]:
AUTHOR_ROOT = "/content/drive/MyDrive/Image beam"
POSITION_ROOT = "/content/drive/MyDrive/scenario23_paper_style_sequence_split"
ZIP_PATH = "/content/drive/MyDrive/scenario23_dev_w_resources.zip"

FULL_CSV_PATH = os.path.join(AUTHOR_ROOT, "scenario23_img_beam.csv")

EXTRACT_ROOT = "/content/scenario23_dev_w_resources"
YOLO_CROP_ROOT = "/content/yolo_uav_crops_scenario23"

INDEX_COL = "index"
IMAGE_COL = "unit1_rgb"
LABEL_COL = "unit1_beam"

print("AUTHOR_ROOT:", os.path.exists(AUTHOR_ROOT), AUTHOR_ROOT)
print("POSITION_ROOT:", os.path.exists(POSITION_ROOT), POSITION_ROOT)
print("ZIP_PATH:", os.path.exists(ZIP_PATH), ZIP_PATH)
print("FULL_CSV_PATH:", os.path.exists(FULL_CSV_PATH), FULL_CSV_PATH)

assert os.path.exists(AUTHOR_ROOT)
assert os.path.exists(POSITION_ROOT)
assert os.path.exists(ZIP_PATH)
assert os.path.exists(FULL_CSV_PATH)

AUTHOR_ROOT: True /content/drive/MyDrive/Image beam
POSITION_ROOT: True /content/drive/MyDrive/scenario23_paper_style_sequence_split
ZIP_PATH: True /content/drive/MyDrive/scenario23_dev_w_resources.zip
FULL_CSV_PATH: True /content/drive/MyDrive/Image beam/scenario23_img_beam.csv


In [6]:
X_pos_train = np.load(os.path.join(POSITION_ROOT, "X_train_seq.npy"))
y_train = np.load(os.path.join(POSITION_ROOT, "y_train_seq.npy"))

X_pos_val = np.load(os.path.join(POSITION_ROOT, "X_val_seq.npy"))
y_val = np.load(os.path.join(POSITION_ROOT, "y_val_seq.npy"))

X_pos_test = np.load(os.path.join(POSITION_ROOT, "X_test_seq.npy"))
y_test = np.load(os.path.join(POSITION_ROOT, "y_test_seq.npy"))

print("Position sequences:")
print("train:", X_pos_train.shape, y_train.shape)
print("val  :", X_pos_val.shape, y_val.shape)
print("test :", X_pos_test.shape, y_test.shape)

assert X_pos_train.shape == (7968, 4, 21)
assert X_pos_val.shape == (2276, 4, 21)
assert X_pos_test.shape == (1139, 4, 21)

num_classes = int(max(y_train.max(), y_val.max(), y_test.max()) + 1)
pos_input_dim = X_pos_train.shape[-1]

print("num_classes:", num_classes)
print("pos_input_dim:", pos_input_dim)

Position sequences:
train: (7968, 4, 21) (7968,)
val  : (2276, 4, 21) (2276,)
test : (1139, 4, 21) (1139,)
num_classes: 29
pos_input_dim: 21


In [7]:
full_img_df = pd.read_csv(FULL_CSV_PATH)
full_img_df = full_img_df.sort_values(INDEX_COL).reset_index(drop=True)

print("Full image CSV:", full_img_df.shape)
print("Columns:", full_img_df.columns.tolist())

for col in [INDEX_COL, IMAGE_COL, LABEL_COL]:
    assert col in full_img_df.columns, f"Missing column: {col}"

display(full_img_df.head())
display(full_img_df.tail())

Full image CSV: (11387, 3)
Columns: ['index', 'unit1_rgb', 'unit1_beam']


,index,unit1_rgb,unit1_beam
0,1,../scenario23_dev/unit1/camera_data/image_BS1_...,22
1,2,../scenario23_dev/unit1/camera_data/image_BS1_...,22
2,3,../scenario23_dev/unit1/camera_data/image_BS1_...,22
3,4,../scenario23_dev/unit1/camera_data/image_BS1_...,22
4,5,../scenario23_dev/unit1/camera_data/image_BS1_...,20


,index,unit1_rgb,unit1_beam
11382,11383,../scenario23_dev/unit1/camera_data/image_BS1_...,20
11383,11384,../scenario23_dev/unit1/camera_data/image_BS1_...,19
11384,11385,../scenario23_dev/unit1/camera_data/image_BS1_...,19
11385,11386,../scenario23_dev/unit1/camera_data/image_BS1_...,19
11386,11387,../scenario23_dev/unit1/camera_data/image_BS1_...,19
